In [34]:
# CSV 데이터를 표 형태로 처리하기 위해 pandas를 가져옵니다.
import pandas as pd

# Hugging Face 비공개 데이터셋에서 파일을 받는 함수를 가져옵니다.
from huggingface_hub import hf_hub_download

# 프로젝트 루트의 .env에서 인증 토큰을 읽는 공통 모듈을 가져옵니다.
from common import secrets

# 프로젝트 루트의 .env에서 본인의 Hugging Face 토큰을 읽습니다.
token, token_source = secrets.load_key(
    ("HUGGINGFACE_ACCESS_TOKEN",),
)

# 토큰이 비어 있다면 잘못된 인증 요청을 보내지 않고 즉시 알려줍니다.
if not token:
    raise RuntimeError(
        "HUGGINGFACE_ACCESS_TOKEN을 찾지 못했습니다. "
        "프로젝트 루트의 .env 파일을 확인해주세요."
    )

# Hugging Face 토큰은 hf_로 시작하는 ASCII 문자열이어야 합니다.
#
# 이 검사를 통해 .env의 한글 주석이나 예시 문장을
# 토큰으로 잘못 읽는 문제를 다운로드 전에 발견합니다.
if not token.startswith("hf_") or not token.isascii():
    raise RuntimeError(
        "Hugging Face 토큰 형식이 올바르지 않습니다. "
        ".env의 HUGGINGFACE_ACCESS_TOKEN 줄에는 실제 토큰만 입력해주세요."
    )

# 팀의 비공개 Hugging Face 데이터셋에서
# 준영님에게 제공된 KOSPI200 피처·라벨 CSV 파일을 받습니다.
#
# 파일은 프로젝트 폴더가 아니라 Hugging Face 캐시에 저장되므로
# CSV 원본이 실수로 GitHub에 커밋되지 않습니다.
data_path = hf_hub_download(
    repo_id="qurious-quant/alphastack-krx-dev",
    filename="small/features_labels_kospi200_dev.csv",
    repo_type="dataset",
    token=token,
)

# 기준일이 숫자로 변형되지 않도록 bas_dd를 문자열로 지정해 CSV를 읽습니다.
df = pd.read_csv(
    data_path,
    dtype={"bas_dd": "string"},
)

# 토큰 전체는 출력하지 않고 어느 파일에서 읽었는지만 확인합니다.
print("토큰 출처:", token_source)

# 데이터의 행 개수와 열 개수를 확인합니다.
print("전체 데이터 크기:", df.shape)

# 모델 입력 X에 넣지 않을 열을 지정합니다.
#
# 날짜·지수 이름은 모델이 학습할 수치 피처가 아니므로 제외합니다.
# 시가·고가·저가·종가 등의 원본값도 현재 기준 모델 피처에서 제외합니다.
# fwd_return_5d와 label은 미래 정보를 담고 있으므로 반드시 X에서 제외합니다.
NOT_FEATURE = {
    "bas_dd",
    "date",
    "index_name",
    "index_class",
    "open",
    "high",
    "low",
    "close",
    "change",
    "change_rate",
    "volume",
    "value",
    "market_cap",
    "fwd_return_5d",
    "label",
}

# 모델 학습에 반드시 필요한 기준일과 정답 열이 존재하는지 확인합니다.
REQUIRED_COLUMNS = {
    "bas_dd",
    "label",
}

# 필수 열 중 데이터에 없는 열을 찾습니다.
missing_columns = REQUIRED_COLUMNS - set(df.columns)

# 필수 열이 없다면 이후 학습을 진행하지 않고 누락된 열을 알려줍니다.
if missing_columns:
    raise ValueError(
        f"필수 열이 없습니다: {sorted(missing_columns)}"
    )

# 제외 대상이 아닌 나머지 열을 모델 입력 피처로 선택합니다.
#
# CSV에 저장된 기존 열 순서를 그대로 유지합니다.
FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in NOT_FEATURE
]

# 선택한 피처들로 모델 입력값 X를 만듭니다.
X = df.loc[:, FEATURE_COLUMNS].copy()

# 미래 5거래일 방향 라벨을 모델의 정답 y로 만듭니다.
#
# 하락=-1, 중립=0, 상승=1입니다.
y = df["label"].copy()

# 선택된 피처 개수와 이름을 확인합니다.
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("피처 개수:", len(FEATURE_COLUMNS))
print("피처 목록:", FEATURE_COLUMNS)

# X에 학습할 행이나 피처가 하나도 없는지 확인합니다.
if X.empty:
    raise ValueError("모델에 사용할 X 데이터가 비어 있습니다.")

# 원본 라벨에 결측값이 있는지 확인합니다.
missing_label_count = int(y.isna().sum())

if missing_label_count > 0:
    raise ValueError(
        f"y에 결측 라벨이 {missing_label_count}개 있습니다."
    )

# CSV의 한글 라벨 앞뒤에 불필요한 공백이 있을 가능성을 제거합니다.
y_text = y.astype("string").str.strip()

# Hugging Face 데이터셋에서 사용하는 한글 라벨을 지정합니다.
EXPECTED_TEXT_LABELS = {
    "하락",
    "중립",
    "상승",
}

# 실제 데이터에 들어 있는 라벨을 확인합니다.
observed_text_labels = set(
    y_text.dropna().unique().tolist()
)

# 정해진 세 라벨에 포함되지 않는 값이 있는지 확인합니다.
unexpected_labels = observed_text_labels - EXPECTED_TEXT_LABELS

if unexpected_labels:
    raise ValueError(
        "알 수 없는 라벨이 포함되어 있습니다: "
        f"{sorted(unexpected_labels)}"
    )

# CSV의 한글 라벨을 모델이 사용할 숫자 라벨로 변환합니다.
#
# 하락=-1
# 중립=0
# 상승=1
LABEL_TO_NUMBER = {
    "하락": -1,
    "중립": 0,
    "상승": 1,
}

# 한글 라벨을 숫자 라벨로 변환합니다.
y = y_text.map(LABEL_TO_NUMBER)

# 변환되지 않은 라벨이 남아 있는지 확인합니다.
#
# 한글 오타나 예상하지 못한 값이 있다면 map() 결과가 결측값이 됩니다.
unmapped_label_count = int(y.isna().sum())

if unmapped_label_count > 0:
    raise ValueError(
        f"숫자로 변환하지 못한 라벨이 {unmapped_label_count}개 있습니다."
    )

# 모든 라벨이 정상적으로 변환된 뒤 정수형으로 변경합니다.
y = y.astype(int)

# 변환된 숫자 라벨의 종류를 확인합니다.
observed_numeric_labels = sorted(y.unique().tolist())

# 프로젝트가 사용하는 세 클래스와 일치하는지 확인합니다.
if observed_numeric_labels != [-1, 0, 1]:
    raise ValueError(
        "숫자 라벨은 하락=-1, 중립=0, 상승=1이어야 합니다. "
        f"현재 라벨: {observed_numeric_labels}"
    )

# X에 문자열처럼 모델이 바로 학습할 수 없는 열이 있는지 확인합니다.
non_numeric_features = [
    column
    for column in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(X[column])
]

if non_numeric_features:
    raise TypeError(
        "숫자가 아닌 피처가 포함되어 있습니다: "
        f"{non_numeric_features}"
    )

# 전체 피처에 포함된 결측값 개수를 계산합니다.
missing_feature_count = int(X.isna().sum().sum())

# 라벨 분포는 사람이 읽기 쉬운 한글 라벨을 기준으로 계산합니다.
#
# reindex를 사용해 항상 하락·중립·상승 순서로 출력되도록 합니다.
label_distribution = (
    y_text.value_counts(normalize=True)
    .reindex(["하락", "중립", "상승"], fill_value=0)
    .mul(100)
    .round(2)
)

# 원본 행은 출력하지 않고 학습에 필요한 요약 정보만 확인합니다.
print("데이터 시작일:", df["bas_dd"].min())
print("데이터 종료일:", df["bas_dd"].max())
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("X 전체 결측값:", missing_feature_count)
print("숫자 라벨:", observed_numeric_labels)
print("라벨 변환:", LABEL_TO_NUMBER)
print("라벨 분포(%):")
print(label_distribution)

토큰 출처: 환경변수 HUGGINGFACE_ACCESS_TOKEN
전체 데이터 크기: (2815, 37)
X 크기: (2815, 22)
y 크기: (2815,)
피처 개수: 22
피처 목록: ['sma_5', 'sma_20', 'sma_60', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_bandwidth', 'true_range', 'atr_14', 'hv_20', 'parkinson_20', 'vol_sma_20', 'vol_ratio_20', 'obv', 'vwap_20', 'vol_roc_5']
데이터 시작일: 20100330
데이터 종료일: 20210823
X 크기: (2815, 22)
y 크기: (2815,)
X 전체 결측값: 0
숫자 라벨: [-1, 0, 1]
라벨 변환: {'하락': -1, '중립': 0, '상승': 1}
라벨 분포(%):
label
하락    27.32
중립    38.69
상승     34.0
Name: proportion, dtype: double[pyarrow]


# Logistic Regression — 피처 조합 B

## 1. 실험 목적

조합 A에서는 `rsi_14`, `bb_bandwidth`, `hv_20`, `vol_ratio_20`을 사용했다. 하지만 변동성과 거래량의 크기를 나타내는 피처가 많고, 상승·하락 방향을 직접 나타내는 피처가 부족해 Logistic Regression이 하락을 한 번도 예측하지 못했다.

조합 B에서는 추세 방향과 모멘텀을 나타내는 피처를 추가해 하락·중립·상승을 구분할 수 있는지 확인한다.

## 2. 입력 피처 X

| 피처 | 계산 방법 | 의미 |
|---|---|---|
| `sma_gap_5_20` | `sma_5 / sma_20 - 1` | 단기·중기 이동평균의 상대적인 차이 |
| `macd_hist_ratio` | `macd_hist / close` | 현재 가격 대비 MACD 히스토그램 |
| `rsi_14` | 기존 RSI 값 | 최근 14거래일의 상승·하락 강도 |
| `hv_20` | 기존 변동성 값 | 최근 20거래일의 과거 변동성 |

### `sma_gap_5_20`

- 0보다 크면 5일 이동평균이 20일 이동평균보다 높은 상태이다.
- 0보다 작으면 5일 이동평균이 20일 이동평균보다 낮은 상태이다.
- 0에 가까우면 단기·중기 이동평균의 차이가 작은 상태이다.

따라서 최근 추세가 상승인지 하락인지 판단하는 데 사용한다.

### `macd_hist_ratio`

MACD 히스토그램은 MACD와 시그널선의 차이이다.

- 양수이면 상승 모멘텀이 상대적으로 강한 상태이다.
- 음수이면 하락 모멘텀이 상대적으로 강한 상태이다.
- 0에 가까우면 상승·하락 모멘텀이 뚜렷하지 않은 상태이다.

KOSPI200의 가격 수준은 시기마다 다르기 때문에 `macd_hist`를 그대로 사용하지 않고 현재 종가로 나누어 가격 대비 비율로 사용한다. 현재 종가 자체는 X에 포함하지 않고 MACD를 정규화하는 계산에만 사용한다.

### `rsi_14`

- RSI가 높으면 최근 상승 움직임이 강했던 상태이다.
- RSI가 낮으면 최근 하락 움직임이 강했던 상태이다.
- RSI가 50 부근이면 상승과 하락의 힘이 비슷한 상태이다.

### `hv_20`

- 값이 크면 최근 가격 변동이 컸던 상태이다.
- 값이 작으면 최근 가격 움직임이 상대적으로 안정적인 상태이다.

추세·모멘텀 피처와 함께 사용해 방향성이 뚜렷한 구간과 중립 구간을 구분할 수 있는지 확인한다.

## 3. 정답 y

- `-1`: 미래 5거래일 하락
- `0`: 미래 5거래일 중립
- `1`: 미래 5거래일 상승

## 4. 검증 방법

- 시간 순서를 유지하는 확장형 워크포워드 검증
- 전체 12개 폴드
- 최초 학습 데이터 750개
- 폴드별 검증 데이터 60개
- 미래 5거래일 라벨을 고려해 학습과 검증 사이 5개 행 제외
- 무작위 분할과 셔플은 사용하지 않음

조합 A와 같은 학습·검증 구간을 사용해 피처 변경에 따른 성능 차이를 비교한다.

In [35]:
# 프로젝트 루트를 sys.path에 넣은 뒤 내부 모듈을 가져와야 합니다.
# ruff: noqa: E402

# 프로젝트 모듈을 불러오기 위해 프로젝트 루트를 찾습니다.
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while (
    project_root != project_root.parent
    and not (project_root / "models").is_dir()
):
    project_root = project_root.parent

if not (project_root / "models").is_dir():
    raise RuntimeError(
        "프로젝트 루트의 models 폴더를 찾지 못했습니다."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 배열과 결과표 계산에 필요한 라이브러리를 가져옵니다.
import numpy as np
import pandas as pd

# 분류 모델의 성능을 계산할 함수를 가져옵니다.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# 프로젝트에서 만든 기준선 예측 함수를 가져옵니다.
from evaluation.baseline import (
    always_up,
    majority_class,
)

# 방향 적중률과 워크포워드 분할 함수를 가져옵니다.
from evaluation.metrics import hit_rate
from evaluation.walk_forward import expanding_splits

# Logistic Regression 모델 생성 함수를 가져옵니다.
from models.logistic import build_logistic_baseline

print("프로젝트 루트:", project_root)

프로젝트 루트: C:\Users\Administrator\Alpha_Stack


In [36]:
# 실험 이름과 조합 B의 피처 이름을 지정합니다.
EXPERIMENT_NAME = "조합 B"

FEATURE_COMBINATION = [
    "sma_gap_5_20",
    "macd_hist_ratio",
    "rsi_14",
    "hv_20",
]

# 워크포워드 실험 설정값을 지정합니다.
N_FOLDS = 12
MIN_TRAIN_SIZE = 750
VALID_SIZE = 60
LABEL_HORIZON = 5

# 결과표에서 사용할 클래스 순서와 이름을 지정합니다.
CLASS_LABELS = [-1, 0, 1]

CLASS_NAMES = {
    -1: "하락",
    0: "중립",
    1: "상승",
}

# 조합 B 계산에 필요한 기존 열을 실수형으로 변환합니다.
close = pd.to_numeric(
    df["close"],
    errors="raise",
).astype(float)

sma_5 = pd.to_numeric(
    df["sma_5"],
    errors="raise",
).astype(float)

sma_20 = pd.to_numeric(
    df["sma_20"],
    errors="raise",
).astype(float)

macd_hist = pd.to_numeric(
    df["macd_hist"],
    errors="raise",
).astype(float)

rsi_14 = pd.to_numeric(
    df["rsi_14"],
    errors="raise",
).astype(float)

hv_20 = pd.to_numeric(
    df["hv_20"],
    errors="raise",
).astype(float)

# 0으로 나누는 문제가 발생하지 않도록 값을 확인합니다.
if (sma_20 == 0).any():
    raise RuntimeError(
        "sma_20에 0이 있어 이동평균 이격률을 계산할 수 없습니다."
    )

if (close == 0).any():
    raise RuntimeError(
        "close에 0이 있어 MACD 비율을 계산할 수 없습니다."
    )

# 단기 이동평균과 중기 이동평균의 상대적 차이를 계산합니다.
#
# 양수: 단기 이동평균이 중기 이동평균보다 높은 상태
# 음수: 단기 이동평균이 중기 이동평균보다 낮은 상태
# 0 근처: 두 이동평균의 차이가 작은 상태
sma_gap_5_20 = (
    sma_5 / sma_20
) - 1.0

# MACD 히스토그램을 현재 가격으로 나눠 정규화합니다.
#
# 양수: 상승 모멘텀이 상대적으로 강한 상태
# 음수: 하락 모멘텀이 상대적으로 강한 상태
# 0 근처: 방향 모멘텀이 뚜렷하지 않은 상태
macd_hist_ratio = (
    macd_hist / close
)

# 조합 B의 네 가지 피처로 X를 만듭니다.
X_selected = pd.DataFrame(
    {
        "sma_gap_5_20": sma_gap_5_20,
        "macd_hist_ratio": macd_hist_ratio,
        "rsi_14": rsi_14,
        "hv_20": hv_20,
    },
    index=df.index,
)

# 무한대 값을 결측값으로 변경해 검사할 수 있게 합니다.
X_selected = X_selected.replace(
    [np.inf, -np.inf],
    np.nan,
)

# 위쪽 데이터 준비 과정에서 만든 y를 정수 배열로 준비합니다.
y_numeric = y.astype(int).to_numpy()

# 데이터가 날짜 오름차순인지 확인합니다.
if not df["bas_dd"].is_monotonic_increasing:
    raise RuntimeError(
        "데이터가 날짜 오름차순으로 정렬되어 있지 않습니다."
    )

# X와 y의 행 개수가 같은지 확인합니다.
if len(X_selected) != len(y_numeric):
    raise RuntimeError(
        "X와 y의 데이터 개수가 서로 다릅니다."
    )

# 조합 B에 결측값이 있는지 확인합니다.
missing_value_count = int(
    X_selected.isna().sum().sum()
)

if missing_value_count != 0:
    raise RuntimeError(
        f"조합 B의 X에 결측값이 "
        f"{missing_value_count}개 있습니다."
    )

print("실험 모델: Logistic Regression")
print("피처 조합:", EXPERIMENT_NAME)
print("사용 피처:", FEATURE_COMBINATION)
print("X 크기:", X_selected.shape)
print("y 크기:", y_numeric.shape)
print("X 전체 결측값:", missing_value_count)
print("숫자 라벨:", sorted(np.unique(y_numeric).tolist()))

display(X_selected.describe().round(6))

실험 모델: Logistic Regression
피처 조합: 조합 B
사용 피처: ['sma_gap_5_20', 'macd_hist_ratio', 'rsi_14', 'hv_20']
X 크기: (2815, 4)
y 크기: (2815,)
X 전체 결측값: 0
숫자 라벨: [-1, 0, 1]


,sma_gap_5_20,macd_hist_ratio,rsi_14,hv_20
count,2815.000000,2815.000000,2815.000000,2815.000000
mean,0.001824,-0.000032,52.440157,0.009732
std,0.021021,0.003831,12.094499,0.004925
min,-0.162245,-0.032638,12.760332,0.003077
25%,-0.009042,-0.002023,44.156156,0.006924
50%,0.002958,-0.000048,52.776217,0.008491
75%,0.013649,0.002114,61.049809,0.010838
max,0.089045,0.018248,85.810765,0.044466


In [37]:
# 확장형 워크포워드 학습·검증 구간을 만듭니다.
#
# horizon=60은 폴드별 검증 데이터 개수입니다.
# label_horizon=5는 y가 미래 5거래일을 사용한다는 의미입니다.
# gap=5를 적용해 학습 라벨과 검증 구간이 겹치지 않게 합니다.
splits = expanding_splits(
    n_samples=len(X_selected),
    n_folds=N_FOLDS,
    min_train=MIN_TRAIN_SIZE,
    horizon=VALID_SIZE,
    gap=LABEL_HORIZON,
    label_horizon=LABEL_HORIZON,
)

# 요청한 12개 폴드가 생성되었는지 확인합니다.
if len(splits) != N_FOLDS:
    raise RuntimeError(
        f"워크포워드 폴드가 {N_FOLDS}개가 아니라 "
        f"{len(splits)}개 생성되었습니다."
    )

# 각 폴드의 시간 순서와 gap을 검사합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    actual_gap = (
        int(valid_index[0])
        - int(train_index[-1])
        - 1
    )

    if train_index[-1] >= valid_index[0]:
        raise RuntimeError(
            f"{fold_number}번 폴드에서 "
            "학습과 검증의 시간 순서가 잘못되었습니다."
        )

    if actual_gap != LABEL_HORIZON:
        raise RuntimeError(
            f"{fold_number}번 폴드의 gap이 "
            f"{actual_gap}개입니다."
        )

print("워크포워드 폴드 수:", len(splits))
print("최초 학습 데이터 수:", len(splits[0][0]))
print("폴드별 검증 데이터 수:", len(splits[0][1]))
print("학습·검증 사이 gap:", LABEL_HORIZON)

워크포워드 폴드 수: 12
최초 학습 데이터 수: 750
폴드별 검증 데이터 수: 60
학습·검증 사이 gap: 5


## 평가 지표

### Accuracy

전체 검증 데이터 중 모델이 하락·중립·상승을 정확하게 맞힌 비율이다. 데이터가 특정 클래스에 치우쳐 있으면 성능이 실제보다 좋아 보일 수 있으므로 Accuracy만 단독으로 사용하지 않는다.

### Macro F1-score

하락·중립·상승의 F1-score를 같은 비중으로 평균한 값이다. 데이터 개수와 관계없이 세 클래스를 동일하게 평가하므로 특정 클래스의 예측을 포기한 모델을 확인하는 데 유용하다.

### Balanced accuracy

하락·중립·상승의 Recall을 같은 비중으로 평균한 값이다. 클래스별 데이터 개수가 다를 때 일반 Accuracy를 보완한다.

### 방향 적중률

실제 중립 데이터를 제외하고 실제 하락·상승 방향을 모델이 맞힌 비율이다. 프로젝트의 `evaluation.metrics.hit_rate()`를 사용한다.

### 항상 상승 기준선

모든 검증 데이터를 상승으로 예측하는 가장 단순한 기준선이다. 모델 성능은 이 기준선과 함께 비교한다.

### 학습 다수 클래스 기준선

현재 폴드의 학습 데이터에서 가장 많았던 클래스를 검증 데이터 전체에 예측하는 기준선이다. 검증 데이터의 정답 분포는 사용하지 않는다.

In [38]:
# 폴드별 평가 결과를 저장할 목록입니다.
fold_results = []

# 전체 OOS 검증 결과를 저장할 목록입니다.
all_y_true = []
all_y_pred = []
all_probabilities = []

# 기준선 예측 결과를 저장할 목록입니다.
all_always_up_pred = []
all_direction_majority_pred = []
all_three_class_majority_pred = []

# 12개 워크포워드 폴드를 시간순으로 학습합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    # 현재 폴드의 학습·검증 데이터를 선택합니다.
    X_train = X_selected.iloc[train_index]
    X_valid = X_selected.iloc[valid_index]

    y_train = y_numeric[train_index]
    y_valid = y_numeric[valid_index]

    # 폴드마다 새로운 Logistic Regression Pipeline을 생성합니다.
    #
    # StandardScaler는 현재 폴드의 학습 데이터만 사용합니다.
    model = build_logistic_baseline()

    # 과거 학습 데이터만 사용해 모델을 학습합니다.
    model.fit(X_train, y_train)

    # 미래 검증 구간의 클래스와 확률을 예측합니다.
    y_pred = model.predict(X_valid)
    probabilities = model.predict_proba(X_valid)

    # Logistic Regression이 세 클래스를 모두 학습했는지 확인합니다.
    classifier = model.named_steps["classifier"]

    if not np.array_equal(
        classifier.classes_,
        np.array(CLASS_LABELS),
    ):
        raise RuntimeError(
            f"{fold_number}번 폴드의 학습 클래스가 "
            f"{classifier.classes_}입니다."
        )

    # 각 검증 행의 예측 확률 합이 1인지 확인합니다.
    if not np.allclose(
        probabilities.sum(axis=1),
        np.ones(len(X_valid)),
    ):
        raise RuntimeError(
            f"{fold_number}번 폴드의 예측 확률 합이 1이 아닙니다."
        )

    # 모든 데이터를 상승으로 예측하는 기준선을 만듭니다.
    always_up_pred = always_up(len(y_valid))

    # 학습 구간의 상승·하락 중 더 많았던 방향을 예측합니다.
    direction_majority_pred = majority_class(
        y_train,
        len(y_valid),
    )

    # 중립을 포함한 세 클래스 중 학습 구간에서
    # 가장 많았던 클래스를 찾습니다.
    train_labels, train_counts = np.unique(
        y_train,
        return_counts=True,
    )

    three_class_majority_label = int(
        train_labels[np.argmax(train_counts)]
    )

    three_class_majority_pred = np.full(
        shape=len(y_valid),
        fill_value=three_class_majority_label,
        dtype=int,
    )

    # 현재 폴드의 모델 성능과 예측 개수를 저장합니다.
    fold_results.append(
        {
            "fold": fold_number,
            "train_size": len(train_index),
            "valid_size": len(valid_index),
            "train_end": df.iloc[train_index[-1]]["bas_dd"],
            "valid_start": df.iloc[valid_index[0]]["bas_dd"],
            "valid_end": df.iloc[valid_index[-1]]["bas_dd"],
            "accuracy": accuracy_score(
                y_valid,
                y_pred,
            ),
            "macro_f1": f1_score(
                y_valid,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid,
                y_pred,
            ),
            "direction_hit_rate": hit_rate(
                y_pred,
                y_valid,
            ),
            "always_up_accuracy": accuracy_score(
                y_valid,
                always_up_pred,
            ),
            "three_class_majority_accuracy": accuracy_score(
                y_valid,
                three_class_majority_pred,
            ),
            "always_up_direction_hit_rate": hit_rate(
                always_up_pred,
                y_valid,
            ),
            "direction_majority_hit_rate": hit_rate(
                direction_majority_pred,
                y_valid,
            ),
            "predicted_down": int(np.sum(y_pred == -1)),
            "predicted_neutral": int(np.sum(y_pred == 0)),
            "predicted_up": int(np.sum(y_pred == 1)),
        }
    )

    # 전체 OOS 평가를 위해 현재 폴드 결과를 저장합니다.
    all_y_true.append(y_valid)
    all_y_pred.append(y_pred)
    all_probabilities.append(probabilities)

    all_always_up_pred.append(always_up_pred)
    all_direction_majority_pred.append(
        direction_majority_pred
    )
    all_three_class_majority_pred.append(
        three_class_majority_pred
    )

    print(
        f"{fold_number:02d}번 폴드 완료 | "
        f"학습 {len(train_index)}개 | "
        f"검증 {len(valid_index)}개 | "
        f"하락 {np.sum(y_pred == -1)}개 | "
        f"중립 {np.sum(y_pred == 0)}개 | "
        f"상승 {np.sum(y_pred == 1)}개"
    )

print("전체 워크포워드 학습이 완료되었습니다.")

01번 폴드 완료 | 학습 750개 | 검증 60개 | 하락 0개 | 중립 1개 | 상승 59개
02번 폴드 완료 | 학습 932개 | 검증 60개 | 하락 0개 | 중립 2개 | 상승 58개
03번 폴드 완료 | 학습 1114개 | 검증 60개 | 하락 0개 | 중립 29개 | 상승 31개
04번 폴드 완료 | 학습 1295개 | 검증 60개 | 하락 0개 | 중립 13개 | 상승 47개
05번 폴드 완료 | 학습 1477개 | 검증 60개 | 하락 0개 | 중립 60개 | 상승 0개
06번 폴드 완료 | 학습 1659개 | 검증 60개 | 하락 0개 | 중립 60개 | 상승 0개
07번 폴드 완료 | 학습 1841개 | 검증 60개 | 하락 0개 | 중립 56개 | 상승 4개
08번 폴드 완료 | 학습 2023개 | 검증 60개 | 하락 0개 | 중립 33개 | 상승 27개
09번 폴드 완료 | 학습 2205개 | 검증 60개 | 하락 0개 | 중립 40개 | 상승 20개
10번 폴드 완료 | 학습 2386개 | 검증 60개 | 하락 8개 | 중립 32개 | 상승 20개
11번 폴드 완료 | 학습 2568개 | 검증 60개 | 하락 0개 | 중립 41개 | 상승 19개
12번 폴드 완료 | 학습 2750개 | 검증 60개 | 하락 0개 | 중립 52개 | 상승 8개
전체 워크포워드 학습이 완료되었습니다.


In [39]:
# 폴드별 결과를 데이터프레임으로 변환합니다.
fold_results_df = pd.DataFrame(fold_results)

# 폴드별 기간, 성능과 예측 클래스 개수를 확인합니다.
display_columns = [
    "fold",
    "train_size",
    "valid_size",
    "train_end",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "balanced_accuracy",
    "direction_hit_rate",
    "predicted_down",
    "predicted_neutral",
    "predicted_up",
]

display(
    fold_results_df.loc[:, display_columns].round(4)
)

,fold,train_size,valid_size,train_end,valid_start,valid_end,accuracy,macro_f1,balanced_accuracy,direction_hit_rate,predicted_down,predicted_neutral,predicted_up
0,1,750,60,20130401,20130409,20130704,0.2833,0.1491,0.3333,0.4595,0,1,59
1,2,932,60,20131224,20140106,20140401,0.2833,0.1740,0.3529,0.5769,0,2,58
2,3,1114,60,20140924,20141002,20141229,0.4000,0.3056,0.3750,0.3125,0,29,31
3,4,1295,60,20150619,20150629,20150921,0.3000,0.2110,0.3684,0.3902,0,13,47
4,5,1477,60,20160316,20160324,20160621,0.4167,0.1961,0.3333,0.0000,0,60,0
5,6,1659,60,20161208,20161216,20170315,0.6000,0.2500,0.3333,0.0000,0,60,0
6,7,1841,60,20170904,20170912,20171212,0.4667,0.2121,0.2917,0.0000,0,56,4
7,8,2023,60,20180607,20180618,20180910,0.4000,0.3025,0.3898,0.2258,0,33,27
8,9,2205,60,20190308,20190318,20190612,0.3500,0.2698,0.3424,0.1622,0,40,20
9,10,2386,60,20191128,20191206,20200305,0.3000,0.2900,0.3202,0.1860,8,32,20


In [40]:
# 12개 폴드의 검증 결과를 하나의 배열로 합칩니다.
oos_y_true = np.concatenate(all_y_true)
oos_y_pred = np.concatenate(all_y_pred)
oos_probabilities = np.vstack(all_probabilities)

# 기준선 결과도 하나의 배열로 합칩니다.
oos_always_up_pred = np.concatenate(
    all_always_up_pred
)

oos_direction_majority_pred = np.concatenate(
    all_direction_majority_pred
)

oos_three_class_majority_pred = np.concatenate(
    all_three_class_majority_pred
)

# 전체 OOS 결과를 요약합니다.
summary = pd.Series(
    {
        "사용 피처 수": len(FEATURE_COMBINATION),
        "워크포워드 폴드 수": len(fold_results_df),
        "전체 OOS 표본 수": len(oos_y_true),
        "평균 accuracy": fold_results_df[
            "accuracy"
        ].mean(),
        "accuracy 표준편차": fold_results_df[
            "accuracy"
        ].std(ddof=1),
        "평균 macro F1": fold_results_df[
            "macro_f1"
        ].mean(),
        "macro F1 표준편차": fold_results_df[
            "macro_f1"
        ].std(ddof=1),
        "전체 OOS accuracy": accuracy_score(
            oos_y_true,
            oos_y_pred,
        ),
        "전체 OOS macro F1": f1_score(
            oos_y_true,
            oos_y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        ),
        "전체 OOS balanced accuracy": (
            balanced_accuracy_score(
                oos_y_true,
                oos_y_pred,
            )
        ),
        "전체 OOS 방향 적중률": hit_rate(
            oos_y_pred,
            oos_y_true,
        ),
        "항상 상승 accuracy": accuracy_score(
            oos_y_true,
            oos_always_up_pred,
        ),
        "3분류 다수 클래스 accuracy": accuracy_score(
            oos_y_true,
            oos_three_class_majority_pred,
        ),
        "항상 상승 방향 적중률": hit_rate(
            oos_always_up_pred,
            oos_y_true,
        ),
        "방향 다수 클래스 적중률": hit_rate(
            oos_direction_majority_pred,
            oos_y_true,
        ),
        "하락 예측 수": int(np.sum(oos_y_pred == -1)),
        "중립 예측 수": int(np.sum(oos_y_pred == 0)),
        "상승 예측 수": int(np.sum(oos_y_pred == 1)),
    },
    name="결과",
)

display(summary.to_frame().round(4))

,결과
사용 피처 수,4.0000
워크포워드 폴드 수,12.0000
전체 OOS 표본 수,720.0000
평균 accuracy,0.3792
accuracy 표준편차,0.1018
평균 macro F1,0.2368
macro F1 표준편차,0.0534
전체 OOS accuracy,0.3792
전체 OOS macro F1,0.2965
전체 OOS balanced accuracy,0.3444


In [41]:
# 실제 클래스별 데이터 개수를 계산합니다.
actual_counts = (
    pd.Series(oos_y_true)
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

# 예측 클래스별 데이터 개수를 계산합니다.
predicted_counts = (
    pd.Series(oos_y_pred)
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

# 실제값과 예측값 분포를 비교합니다.
prediction_distribution_df = pd.DataFrame(
    {
        "클래스": [
            f"{CLASS_NAMES[label]}({label})"
            for label in CLASS_LABELS
        ],
        "실제 개수": actual_counts.to_numpy(),
        "예측 개수": predicted_counts.to_numpy(),
    }
)

display(prediction_distribution_df)

,클래스,실제 개수,예측 개수
0,하락(-1),204,8
1,중립(0),306,419
2,상승(1),210,293


In [42]:
# 전체 OOS 데이터에서 클래스별 예측 확률을 확인합니다.
probability_columns = [
    f"{CLASS_NAMES[label]}({label})"
    for label in CLASS_LABELS
]

oos_probability_df = pd.DataFrame(
    oos_probabilities,
    columns=probability_columns,
)

# 각 클래스의 평균·최대 확률과 최종 선택 횟수를 계산합니다.
probability_summary = pd.DataFrame(
    {
        "평균 예측 확률": (
            oos_probability_df.mean()
        ),
        "최대 예측 확률": (
            oos_probability_df.max()
        ),
        "가장 높은 확률로 선택된 횟수": (
            oos_probability_df.idxmax(axis=1)
            .value_counts()
            .reindex(
                probability_columns,
                fill_value=0,
            )
        ),
    }
)

display(probability_summary.round(4))

,평균 예측 확률,최대 예측 확률,가장 높은 확률로 선택된 횟수
하락(-1),0.2685,0.4250,8
중립(0),0.3833,0.5522,419
상승(1),0.3482,0.5842,293


In [43]:
# 혼동행렬에 표시할 클래스 이름을 지정합니다.
display_class_names = [
    "하락(-1)",
    "중립(0)",
    "상승(1)",
]

# 전체 OOS 예측 결과의 혼동행렬을 계산합니다.
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
    ),
    index=[
        f"실제 {name}"
        for name in display_class_names
    ],
    columns=[
        f"예측 {name}"
        for name in display_class_names
    ],
)

display(confusion_matrix_df)

,예측 하락(-1),예측 중립(0),예측 상승(1)
실제 하락(-1),4,123,77
실제 중립(0),1,179,126
실제 상승(1),3,117,90


In [44]:
# 하락·중립·상승의 Precision, Recall, F1-score를 계산합니다.
classification_report_df = pd.DataFrame(
    classification_report(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
        target_names=display_class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(classification_report_df.round(4))

,precision,recall,f1-score,support
하락(-1),0.5000,0.0196,0.0377,204.0000
중립(0),0.4272,0.5850,0.4938,306.0000
상승(1),0.3072,0.4286,0.3579,210.0000
accuracy,0.3792,0.3792,0.3792,0.3792
macro avg,0.4115,0.3444,0.2965,720.0000
weighted avg,0.4128,0.3792,0.3249,720.0000


# Logistic Regression 조합 B 실험 결과 해석

## 1. 전체 실험 결과

| 평가 항목 | 결과 |
|---|---:|
| 사용 피처 수 | 4개 |
| 워크포워드 폴드 수 | 12개 |
| 전체 OOS 표본 수 | 720개 |
| 평균 Accuracy | 0.3792 |
| Accuracy 표준편차 | 0.1018 |
| 평균 Macro F1 | 0.2368 |
| Macro F1 표준편차 | 0.0534 |
| 전체 OOS Accuracy | 0.3792 |
| 전체 OOS Macro F1 | 0.2965 |
| 전체 OOS Balanced Accuracy | 0.3444 |
| 전체 OOS 방향 적중률 | 0.2271 |

전체 OOS 720개 중 273개를 올바르게 예측해 Accuracy는 약 37.92%로 나타났다.

Accuracy 표준편차는 0.1018이다. 폴드마다 Accuracy가 평균을 중심으로 약 10%포인트 정도 달라졌다는 의미이므로, 시기에 따라 모델 성능이 비교적 크게 변한 것으로 볼 수 있다.

평균 Macro F1은 12개 폴드에서 각각 계산한 Macro F1을 평균한 결과이고, 전체 OOS Macro F1은 12개 검증 구간의 720개 예측을 하나로 합쳐 계산한 결과이다. Macro F1은 단순 합산 지표가 아니므로 두 값이 서로 다르게 나올 수 있다.

폴드별 평균 Macro F1이 0.2368로 전체 OOS Macro F1 0.2965보다 낮게 나타났다는 것은 일부 폴드에서 특정 클래스를 제대로 예측하지 못해 성능이 크게 낮아졌다는 것을 보여준다.

## 2. 기준선 비교

| 모델 또는 기준선 | 결과 |
|---|---:|
| Logistic Regression Accuracy | 0.3792 |
| 항상 상승 Accuracy | 0.2917 |
| 3분류 다수 클래스 Accuracy | 0.3903 |
| Logistic Regression 방향 적중률 | 0.2271 |
| 항상 상승 방향 적중률 | 0.5072 |
| 방향 다수 클래스 적중률 | 0.5072 |

Logistic Regression의 Accuracy 0.3792는 항상 상승 기준선 0.2917보다는 높았다. 그러나 학습 구간의 다수 클래스를 예측하는 기준선 0.3903보다는 약 1.11%포인트 낮았다.

방향 적중률에서는 Logistic Regression이 0.2271을 기록했다. 이는 항상 상승 기준선과 방향 다수 클래스 기준선의 0.5072보다 크게 낮은 결과이다.

방향 적중률은 실제 중립 데이터를 제외하고 실제 하락과 상승을 얼마나 맞혔는지 계산한다. 현재 모델은 실제 하락·상승 데이터 414개 중 94개만 올바른 방향으로 예측했다.

따라서 조합 B의 Logistic Regression은 단순 기준선보다 우수한 방향 예측 성능을 보여주지 못했다.

## 3. 실제값과 예측값 분포

| 클래스 | 실제 개수 | 예측 개수 |
|---|---:|---:|
| 하락(-1) | 204 | 8 |
| 중립(0) | 306 | 419 |
| 상승(1) | 210 | 293 |

실제 하락 데이터는 204개였지만 모델은 하락을 8번만 예측했다. 조합 A에서는 하락을 한 번도 예측하지 않았기 때문에 조합 B에서 하락 예측 자체는 새롭게 나타났다.

하지만 실제 하락 데이터 수에 비해 하락 예측 수가 매우 적어 하락 구간을 충분히 구분했다고 보기는 어렵다.

모델은 중립을 419번 예측해 실제 중립 306개보다 113번 많이 예측했다. 상승도 실제 210개보다 많은 293번을 예측했다. 결과적으로 현재 모델은 대부분의 데이터를 중립 또는 상승으로 분류했다.

## 4. 클래스별 예측 확률

| 클래스 | 평균 예측 확률 | 최대 예측 확률 | 최종 선택 횟수 |
|---|---:|---:|---:|
| 하락(-1) | 0.2685 | 0.4250 | 8 |
| 중립(0) | 0.3833 | 0.5522 | 419 |
| 상승(1) | 0.3482 | 0.5842 | 293 |

하락의 평균 예측 확률은 0.2685로 0이 아니었다. 따라서 모델이 하락 클래스를 학습하지 못했거나 코드에서 하락 라벨이 빠진 것은 아니다.

Logistic Regression은 각 데이터에서 하락·중립·상승 확률을 계산한 뒤 가장 높은 확률을 가진 클래스를 최종 예측으로 선택한다. 하락 확률은 계산됐지만 대부분 중립 또는 상승 확률보다 낮았기 때문에 최종 하락 예측은 8번만 발생했다.

중립의 평균 예측 확률이 0.3833으로 가장 높았고, 상승이 0.3482, 하락이 0.2685로 나타났다. 모델이 전반적으로 하락보다 중립과 상승을 선택하는 경향을 보였다는 것을 확인할 수 있다.

## 5. 혼동행렬

| 실제값 \ 예측값 | 하락(-1) | 중립(0) | 상승(1) |
|---|---:|---:|---:|
| 실제 하락(-1) | 4 | 123 | 77 |
| 실제 중립(0) | 1 | 179 | 126 |
| 실제 상승(1) | 3 | 117 | 90 |

혼동행렬에서 행은 실제 정답이고 열은 모델의 예측이다. 대각선의 값은 모델이 올바르게 예측한 개수이다.

- 실제 하락 204개 중 4개만 하락으로 맞혔다.
- 실제 하락 중 123개를 중립, 77개를 상승으로 잘못 예측했다.
- 실제 중립 306개 중 179개를 중립으로 맞혔다.
- 실제 상승 210개 중 90개를 상승으로 맞혔다.
- 전체 720개 중 `4 + 179 + 90 = 273개`를 올바르게 예측했다.

조합 B는 하락을 일부 예측했지만 실제 하락 대부분을 중립 또는 상승으로 잘못 분류했다.

## 6. 클래스별 평가 지표

| 클래스 | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| 하락(-1) | 0.5000 | 0.0196 | 0.0377 | 204 |
| 중립(0) | 0.4272 | 0.5850 | 0.4938 | 306 |
| 상승(1) | 0.3072 | 0.4286 | 0.3579 | 210 |

### 하락(-1)

하락 Precision은 0.5000이다. 모델이 하락으로 예측한 8개 중 4개가 실제 하락이었다.

그러나 하락 Recall은 0.0196으로, 실제 하락 204개 중 약 1.96%만 찾아냈다. Precision 0.5000은 하락 예측이 8개뿐인 매우 작은 표본에서 계산된 값이므로 이것만 보고 하락 성능이 좋다고 판단하면 안 된다.

하락 F1-score는 0.0377로 매우 낮았다.

### 중립(0)

중립 Recall은 0.5850으로 실제 중립 306개 중 약 58.50%를 찾아냈다. 중립 F1-score는 0.4938로 세 클래스 중 가장 높았다.

현재 모델은 세 클래스 중 중립을 상대적으로 가장 잘 예측했다.

### 상승(1)

상승 Recall은 0.4286으로 실제 상승 210개 중 약 42.86%를 찾아냈다. 상승 F1-score는 0.3579로 나타났다.

상승은 하락보다 잘 구분했지만, 중립을 상승으로 잘못 예측한 경우가 126개로 많았다.

## 7. 조합 A와 조합 B 비교

| 평가 항목 | 조합 A | 조합 B | 변화 |
|---|---:|---:|---:|
| 전체 OOS Accuracy | 0.3958 | 0.3792 | -0.0166 |
| 전체 OOS Macro F1 | 0.2971 | 0.2965 | -0.0006 |
| 하락 예측 수 | 0 | 8 | +8 |
| 하락 Recall | 0.0000 | 0.0196 | +0.0196 |

조합 B는 방향성 피처를 추가하면서 조합 A와 달리 하락을 8번 예측했다. 하지만 실제 하락 204개 중 4개만 맞혀 하락 Recall은 1.96%에 그쳤다.

전체 OOS Accuracy는 조합 A보다 약 1.66%포인트 낮아졌고, Macro F1도 거의 개선되지 않았다. 따라서 조합 B는 조합 A의 하락 예측 0개 문제를 일부 완화했지만 실질적인 분류 성능 개선으로 이어지지는 않았다.

## 8. 최종 판단

조합 B는 조합 A보다 방향성을 나타내는 피처를 강화했지만 다음 문제가 남았다.

- 하락 204개 중 4개만 예측했다.
- 3분류 다수 클래스 Accuracy보다 낮았다.
- 방향 적중률이 항상 상승 기준선보다 낮았다.
- 폴드별 Accuracy 변동이 비교적 컸다.
- 조합 A와 비교해 전체 Accuracy와 Macro F1이 개선되지 않았다.

따라서 조합 B를 Logistic Regression의 최종 피처 조합으로 선택하기 어렵다.

다만 조합 A와 B의 결과만으로 Logistic Regression 전체가 사용할 수 없는 모델이라고 단정할 수는 없다. 현재 결과는 기본 Logistic Regression이 클래스 불균형과 피처 간 관계를 충분히 반영하지 못했을 가능성을 보여준다.

조합 A와 조합 B의 결과는 삭제하지 않고 실험 기록으로 유지한다. 다음 실험에서는 한 번에 여러 조건을 변경하지 않고, 피처 조합 또는 클래스 가중치 중 하나만 변경해 성능 변화의 원인을 확인한다.